# 💼 Pay Equity Lens — Business Recommendations

Based on EDA findings and model results, this notebook translates data insights into actionable HR recommendations.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/processed/hr_with_predictions.csv')
print(f"Loaded: {df.shape}")

plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d2e',
    'axes.labelcolor': 'white', 'xtick.color': 'white', 'ytick.color': 'white',
    'text.color': 'white', 'axes.titlecolor': 'white', 'axes.edgecolor': '#333355', 'grid.color': '#333355',
})

## 1. Pay Equity Summary Dashboard

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0f1117')

underpaid_gender = df.groupby('Gender')['Underpaid'].mean() * 100
bars = axes[0].bar(underpaid_gender.index, underpaid_gender.values,
                   color=['#E07B54', '#54A0E0'], edgecolor='#0f1117', width=0.5)
for bar, val in zip(bars, underpaid_gender.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}%', ha='center', color='white', fontweight='bold')
axes[0].set_title('% Underpaid by Gender\n(>$500 below predicted fair salary)', fontweight='bold')
axes[0].set_ylabel('% Underpaid')
axes[0].grid(True, alpha=0.3, axis='y')

dept_underpaid = df.groupby('Department')['Underpaid'].mean() * 100
bars2 = axes[1].bar(dept_underpaid.index, dept_underpaid.values,
                    color=['#7B5EA7', '#5EC87B', '#E0C454'], edgecolor='#0f1117', width=0.5)
for bar, val in zip(bars2, dept_underpaid.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.1f}%', ha='center', color='white', fontweight='bold')
axes[1].set_title('% Underpaid by Department', fontweight='bold')
axes[1].set_ylabel('% Underpaid')
axes[1].tick_params(axis='x', rotation=10)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../reports/fig6_pay_equity.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()

## Recommendations

1. Run a formal pay audit — 16.9% of female employees are underpaid vs benchmark
2. Create transparent salary bands per role/level using model predictions as midpoint
3. HR department needs urgent pay correction (21.6% underpaid)
4. Sales base-pay needs review separately from commission
5. Run this model quarterly and flag employees dipping >10% below predicted salary

In [ ]:
# Top 20 most underpaid employees (for HR action list)
underpaid_df = df[df['Underpaid'] == True][['Gender', 'Department', 'JobRole', 'JobLevelLabel', 
                                             'MonthlyIncome', 'PredictedFairSalary', 'SalaryGap']]
underpaid_df = underpaid_df.sort_values('SalaryGap').head(20)
underpaid_df['SalaryGap'] = underpaid_df['SalaryGap'].round(0).astype(int)
print("Top 20 Most Underpaid Employees:")
print(underpaid_df.to_string(index=False))

## Conclusion

The analysis confirms a 9.2% gender pay gap that cannot be explained by experience or education alone. The Random Forest model (R²=0.93) provides a reliable fair salary benchmark that HR can use to prioritize corrections.